In [1]:
import numpy as np
import tensorflow as tf

In [2]:
from tensorflow.compat.v1 import ConfigProto
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
config=tf.compat.v1.ConfigProto()
config.gpu_options.per_process_gpu_memory_fraction = 0.8
config.gpu_options.allow_growth=True
sess=tf.compat.v1.Session(config=config) 

In [3]:
from network import build_resnet18
from train import WarmUpCosine, CustomWeightDecaySGD, LastNSaver, make_train_ds, make_test_ds, RSquared 

In [4]:
num_classes = 3
initial_lr = 0.1
weight_decay = 1e-4
epochs = 200
warmup_epochs = 5
batch_size = 128
image_size = 32

In [ ]:
SAVE_PATH = "utkface_12k_split.npz" 
def load_dataset_if_exists(path=SAVE_PATH):
    """
    Load if file exists, otherwise return None.
    """
    if os.path.exists(path):
        data = np.load(path)
        print("✔ Cached data detected, loading directly")
        return (data["x_train"], data["y_train"],
                data["x_test"], data["y_test"])
    return None

In [6]:
x_train, y_train, x_test, y_test = load_dataset_if_exists()
y_train = tf.cast(y_train, dtype=tf.float32)
y_test = tf.cast(y_test, dtype=tf.float32)

✔ 检测到缓存数据，已直接加载


In [7]:
NN_18=[32,32,32,64,64,128,128,256,256]

In [8]:
model = build_resnet18(NN_18, num_classes=1)

In [9]:
total_steps = epochs * (x_train.shape[0] // batch_size)
warmup_steps = warmup_epochs * (x_train.shape[0] // batch_size)
lr_schedule = WarmUpCosine(initial_lr, total_steps, warmup_steps)
optimizer = CustomWeightDecaySGD(
    weight_decay=weight_decay,
    learning_rate=lr_schedule,
    momentum=0.9,
    nesterov=True
)
model.compile(
    optimizer=optimizer,
    loss='mse',
    metrics=[RSquared()]
)

In [10]:
saver = LastNSaver(n=20)

In [11]:
train_ds = make_train_ds(
    x_train, y_train,
    batch_size=batch_size,
    image_size=32, pad=4,
    mixup_alpha=0.05   # 0.2~0.4 common; increase if overfitting
)
test_ds = make_test_ds(x_test, y_test, batch_size=batch_size)

model.fit(
    train_ds,
    epochs=epochs,
    validation_data=test_ds,
    verbose=2,
    callbacks=[saver]
)

Epoch 1/200
156/156 - 10s - loss: 0.0337 - r_squared: -1.1064e-01 - val_loss: 0.0600 - val_r_squared: -8.3590e-01 - 10s/epoch - 61ms/step
Epoch 2/200
156/156 - 4s - loss: 0.0246 - r_squared: 0.1734 - val_loss: 0.0254 - val_r_squared: 0.2236 - 4s/epoch - 24ms/step
Epoch 3/200
156/156 - 4s - loss: 0.0206 - r_squared: 0.3117 - val_loss: 0.0220 - val_r_squared: 0.3275 - 4s/epoch - 24ms/step
Epoch 4/200
156/156 - 4s - loss: 0.0180 - r_squared: 0.4003 - val_loss: 0.0289 - val_r_squared: 0.1167 - 4s/epoch - 25ms/step
Epoch 5/200
156/156 - 4s - loss: 0.0156 - r_squared: 0.4853 - val_loss: 0.0191 - val_r_squared: 0.4165 - 4s/epoch - 24ms/step
Epoch 6/200
156/156 - 4s - loss: 0.0135 - r_squared: 0.5513 - val_loss: 0.0255 - val_r_squared: 0.2204 - 4s/epoch - 24ms/step
Epoch 7/200
156/156 - 4s - loss: 0.0124 - r_squared: 0.5908 - val_loss: 0.0158 - val_r_squared: 0.5166 - 4s/epoch - 24ms/step
Epoch 8/200
156/156 - 4s - loss: 0.0112 - r_squared: 0.6303 - val_loss: 0.0365 - val_r_squared: -1.1601e-0

In [12]:
model.save("Res_18.h5")

C:\Users\user\.conda\envs\tf\lib\site-packages\keras\engine\functional.py:1410: CustomMaskWarning: Custom mask layers require a config and must override get_config. When loading, the custom mask layer must be passed to the custom_objects argument.
  layer_config = serialize_layer_fn(layer)
